## Summary

This notebook generated thesis-ready result tables from the final experimental comparison results. The tables summarize overall detection performance, model size reduction, inference time improvement, and the best-performing model under different evaluation criteria. The results show that dynamic quantization provides the strongest deployment efficiency benefits, while pruning can preserve or slightly improve detection performance depending on the dataset.

In [7]:
table_files = [
    THESIS_FINAL_TABLE_PATH,
    SIZE_REDUCTION_TABLE_PATH,
    INFERENCE_IMPROVEMENT_TABLE_PATH,
    BEST_MODEL_TABLE_PATH
]

for table in table_files:
    print(table.name, table.exists())

thesis_final_model_comparison_table.csv True
thesis_model_size_reduction_table.csv True
thesis_inference_improvement_table.csv True
thesis_best_model_summary_table.csv True


In [6]:
best_rows = []

for dataset in results["Dataset"].unique():
    dataset_results = results[results["Dataset"] == dataset].copy()

    best_f1 = dataset_results.loc[dataset_results["F1-score"].idxmax()]
    smallest_model = dataset_results.loc[dataset_results["Model Size MB"].idxmin()]
    fastest_model = dataset_results.loc[dataset_results["Inference Time Seconds"].idxmin()]

    best_rows.append({
        "Dataset": dataset,
        "Best F1-score Model": best_f1["Model"],
        "Best F1-score (%)": round(best_f1["F1-score"] * 100, 2),
        "Smallest Model": smallest_model["Model"],
        "Smallest Model Size MB": round(smallest_model["Model Size MB"], 3),
        "Fastest Inference Model": fastest_model["Model"],
        "Fastest Inference Time Seconds": round(fastest_model["Inference Time Seconds"], 3)
    })

best_model_summary = pd.DataFrame(best_rows)

best_model_summary.to_csv(BEST_MODEL_TABLE_PATH, index=False)

print("Saved:", BEST_MODEL_TABLE_PATH)
display(best_model_summary)

Saved: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\thesis_best_model_summary_table.csv


,Dataset,Best F1-score Model,Best F1-score (%),Smallest Model,Smallest Model Size MB,Fastest Inference Model,Fastest Inference Time Seconds
0,NSL-KDD,Dynamic Quantized TFLite CNN,74.95,Dynamic Quantized TFLite CNN,0.046,Dynamic Quantized TFLite CNN,0.579
1,UNSW-NB15,50% Magnitude Pruned CNN,94.69,Dynamic Quantized TFLite CNN,0.050,Dynamic Quantized TFLite CNN,3.677


In [5]:
baseline_inference = results[results["Model"] == "Baseline CNN"][
    ["Dataset", "Inference Time Seconds"]
].rename(columns={"Inference Time Seconds": "Baseline Inference Time Seconds"})

optimized_inference = results[results["Model"] != "Baseline CNN"][
    ["Dataset", "Model", "Inference Time Seconds"]
].rename(columns={"Inference Time Seconds": "Optimized Inference Time Seconds"})

inference_improvement_table = optimized_inference.merge(
    baseline_inference,
    on="Dataset",
    how="left"
)

inference_improvement_table["Inference Time Change Seconds"] = (
    inference_improvement_table["Baseline Inference Time Seconds"]
    - inference_improvement_table["Optimized Inference Time Seconds"]
)

inference_improvement_table["Inference Time Improvement (%)"] = (
    inference_improvement_table["Inference Time Change Seconds"]
    / inference_improvement_table["Baseline Inference Time Seconds"]
    * 100
)

inference_improvement_table = inference_improvement_table[
    [
        "Dataset",
        "Model",
        "Baseline Inference Time Seconds",
        "Optimized Inference Time Seconds",
        "Inference Time Change Seconds",
        "Inference Time Improvement (%)"
    ]
]

inference_improvement_table = inference_improvement_table.round({
    "Baseline Inference Time Seconds": 3,
    "Optimized Inference Time Seconds": 3,
    "Inference Time Change Seconds": 3,
    "Inference Time Improvement (%)": 2
})

inference_improvement_table.to_csv(INFERENCE_IMPROVEMENT_TABLE_PATH, index=False)

print("Saved:", INFERENCE_IMPROVEMENT_TABLE_PATH)
display(inference_improvement_table)

Saved: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\thesis_inference_improvement_table.csv


,Dataset,Model,Baseline Inference Time Seconds,Optimized Inference Time Seconds,Inference Time Change Seconds,Inference Time Improvement (%)
0,NSL-KDD,Dynamic Quantized TFLite CNN,2.750,0.579,2.171,78.95
1,UNSW-NB15,Dynamic Quantized TFLite CNN,14.661,3.677,10.984,74.92
2,NSL-KDD,50% Magnitude Pruned CNN,2.750,2.601,0.149,5.42
3,UNSW-NB15,50% Magnitude Pruned CNN,14.661,14.605,0.056,0.38


In [4]:
baseline_sizes = results[results["Model"] == "Baseline CNN"][
    ["Dataset", "Model Size MB"]
].rename(columns={"Model Size MB": "Baseline Size MB"})

optimized_sizes = results[results["Model"] != "Baseline CNN"][
    ["Dataset", "Model", "Model Size MB"]
].rename(columns={"Model Size MB": "Optimized Size MB"})

size_reduction_table = optimized_sizes.merge(
    baseline_sizes,
    on="Dataset",
    how="left"
)

size_reduction_table["Size Reduction MB"] = (
    size_reduction_table["Baseline Size MB"] - size_reduction_table["Optimized Size MB"]
)

size_reduction_table["Size Reduction (%)"] = (
    size_reduction_table["Size Reduction MB"] / size_reduction_table["Baseline Size MB"] * 100
)

size_reduction_table = size_reduction_table[
    [
        "Dataset",
        "Model",
        "Baseline Size MB",
        "Optimized Size MB",
        "Size Reduction MB",
        "Size Reduction (%)"
    ]
]

size_reduction_table = size_reduction_table.round({
    "Baseline Size MB": 3,
    "Optimized Size MB": 3,
    "Size Reduction MB": 3,
    "Size Reduction (%)": 2
})

size_reduction_table.to_csv(SIZE_REDUCTION_TABLE_PATH, index=False)

print("Saved:", SIZE_REDUCTION_TABLE_PATH)
display(size_reduction_table)

Saved: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\thesis_model_size_reduction_table.csv


,Dataset,Model,Baseline Size MB,Optimized Size MB,Size Reduction MB,Size Reduction (%)
0,NSL-KDD,Dynamic Quantized TFLite CNN,0.489,0.046,0.442,90.52
1,UNSW-NB15,Dynamic Quantized TFLite CNN,0.536,0.050,0.485,90.62
2,NSL-KDD,50% Magnitude Pruned CNN,0.489,0.184,0.305,62.40
3,UNSW-NB15,50% Magnitude Pruned CNN,0.536,0.199,0.336,62.77


In [3]:
thesis_table = results.copy()

for col in ["Accuracy", "Precision", "Recall", "F1-score"]:
    thesis_table[col] = (thesis_table[col] * 100).round(2)

if "Training Time Seconds" in thesis_table.columns:
    thesis_table["Training Time Seconds"] = thesis_table["Training Time Seconds"].round(2)

thesis_table["Inference Time Seconds"] = thesis_table["Inference Time Seconds"].round(3)
thesis_table["Model Size MB"] = thesis_table["Model Size MB"].round(3)

if "Sparsity" in thesis_table.columns:
    thesis_table["Sparsity"] = (thesis_table["Sparsity"] * 100).round(2)

thesis_table.to_csv(THESIS_FINAL_TABLE_PATH, index=False)

print("Saved:", THESIS_FINAL_TABLE_PATH)
display(thesis_table)

Saved: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\thesis_final_model_comparison_table.csv


,Dataset,Model,Accuracy,Precision,Recall,F1-score,Training Time Seconds,Inference Time Seconds,Model Size MB,Sparsity
0,NSL-KDD,Baseline CNN,76.61,96.02,61.45,74.94,137.07,2.750,0.489,NaN
1,UNSW-NB15,Baseline CNN,92.23,96.10,92.33,94.18,43.73,14.661,0.536,NaN
2,NSL-KDD,Dynamic Quantized TFLite CNN,76.62,96.04,61.46,74.95,NaN,0.579,0.046,NaN
3,UNSW-NB15,Dynamic Quantized TFLite CNN,92.23,96.10,92.33,94.18,NaN,3.677,0.050,NaN
4,NSL-KDD,50% Magnitude Pruned CNN,76.48,96.26,61.05,74.71,NaN,2.601,0.184,50.0
5,UNSW-NB15,50% Magnitude Pruned CNN,92.82,95.47,93.92,94.69,NaN,14.605,0.199,50.0


In [2]:
results = pd.read_csv(ALL_RESULTS_PATH)

display(results)

,Dataset,Model,Accuracy,Precision,Recall,F1-score,Training Time Seconds,Inference Time Seconds,Model Size MB,Sparsity
0,NSL-KDD,Baseline CNN,0.766057,0.960185,0.614509,0.749406,137.072274,2.750228,0.488823,NaN
1,UNSW-NB15,Baseline CNN,0.922289,0.961005,0.923287,0.941769,43.730621,14.660965,0.535698,NaN
2,NSL-KDD,Dynamic Quantized TFLite CNN,0.766191,0.960424,0.614587,0.749537,NaN,0.578795,0.046349,NaN
3,UNSW-NB15,Dynamic Quantized TFLite CNN,0.922300,0.961038,0.923270,0.941776,NaN,3.677338,0.050255,NaN
4,NSL-KDD,50% Magnitude Pruned CNN,0.764771,0.962644,0.610457,0.747127,NaN,2.601195,0.183807,0.5
5,UNSW-NB15,50% Magnitude Pruned CNN,0.928243,0.954661,0.939174,0.946854,NaN,14.604850,0.199432,0.5


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()

RESULTS_DIR = PROJECT_ROOT / "results"

ALL_RESULTS_PATH = RESULTS_DIR / "all_model_comparison_results.csv"

THESIS_FINAL_TABLE_PATH = RESULTS_DIR / "thesis_final_model_comparison_table.csv"
SIZE_REDUCTION_TABLE_PATH = RESULTS_DIR / "thesis_model_size_reduction_table.csv"
INFERENCE_IMPROVEMENT_TABLE_PATH = RESULTS_DIR / "thesis_inference_improvement_table.csv"
BEST_MODEL_TABLE_PATH = RESULTS_DIR / "thesis_best_model_summary_table.csv"

print("All model results exist:", ALL_RESULTS_PATH.exists())

All model results exist: True
